# PATH CONFIG

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

%load_ext autoreload
%autoreload 2
# %matplotlib inline

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/notebooks
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/app


In [2]:
from src.config import Configuration

CONFIG = Configuration(
    model_name="meta-llama/Llama-2-7b-hf",
    src_code = "spa_Latn",
    tgt_code = "epo_Latn",
    data_fraction=0.03,

    num_shots = 5,

    max_tok_length = 32,
    batch_size = 24,
)

# Tokenizer

In [3]:
import os
import dotenv
from huggingface_hub import login

dotenv.load_dotenv()
login(token=os.getenv("HUGGING_FACE_TOKEN"))

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
python-dotenv could not parse statement starting at line 2
python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 8


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG.model_name,
    use_auth_token=True, 
    padding=True, 
    pad_to_multiple_of=8, 
    # src_lang=CONFIG.src_code, 
    # tgt_lang=CONFIG.tgt_code, 
    truncation=True, 
    max_length=CONFIG.max_tok_length,
    padding_side='left',
)
tokenizer.pad_token = tokenizer.eos_token


/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/transformers/models/auto/tokenization_auto.py:1025: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


# Dataset

In [5]:
from src.data import tokenize_dataframe

df_corpus_clean = pd.read_csv(CONFIG.corpus_path)
df_corpus_clean.rename(columns={
    CONFIG.src_name: CONFIG.src_code, 
    CONFIG.tgt_name: CONFIG.tgt_code
}, inplace=True)

tokenized_corpus = tokenize_dataframe(
    df_corpus_clean,
    CONFIG.src_code,
    CONFIG.tgt_code,
    tokenizer
)
tokenized_corpus.to_pickle(CONFIG.tokenized_corpus_llama_prompting)


100%|██████████| 287091/287091 [00:16<00:00, 17760.43it/s]


In [6]:
tokenized_corpus = pd.read_pickle(CONFIG.tokenized_corpus_llama_prompting)

In [7]:
tokenized_corpus.iloc[0]

{'input_ids': [1, 28534, 2611, 29874, 2002, 8005, 23517, 14859, 2966, 359, 29991], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [1, 1277, 544, 11344, 2420, 21552, 413, 329, 326, 3848, 29876, 29991]}

In [8]:
n_total = len(tokenized_corpus)

mask = tokenized_corpus.apply(
    lambda x: 
        len(x["input_ids"]) <= CONFIG.max_tok_length and 
        len(x["labels"]) <= CONFIG.max_tok_length, 
)

tokenized_filter = tokenized_corpus[mask]
n_filtered = len(tokenized_filter)
print(f"Total tokenized samples: {n_total:_}")
print(f"Total tokenized samples after filtering: {n_filtered:_}")

Total tokenized samples: 287_091
Total tokenized samples after filtering: 189_701


In [9]:
# Shuffle the dataframe first
tokenized_filter = tokenized_filter.sample(frac=1, random_state=CONFIG.seed).reset_index(drop=True)

n_test = int(n_filtered * CONFIG.test_split)
n_val = int(n_filtered * CONFIG.val_split)

tok_test = tokenized_filter[:n_test].reset_index(drop=True)
tok_val = tokenized_filter[n_test:n_test + n_val].reset_index(drop=True)
tok_train = tokenized_filter[n_test + n_val:].reset_index(drop=True)

print(f"Dataset sizes:")
print(f"  Train: {len(tok_train):_} ({len(tok_train)/n_filtered:.2%})")
print(f"  Val:   {len(tok_val):_} ({len(tok_val)/n_filtered:.2%})")
print(f"  Test:  {len(tok_test):_} ({len(tok_test)/n_filtered:.2%})")
print(f"  Total: {n_filtered:_}")

Dataset sizes:
  Train: 132_791 (70.00%)
  Val:   28_455 (15.00%)
  Test:  28_455 (15.00%)
  Total: 189_701


Reduce dataset size for experiments

In [10]:
n_train = int(len(tok_train)*CONFIG.data_fraction)
n_val = int(len(tok_val)*CONFIG.data_fraction)
n_test = int(len(tok_test)*CONFIG.data_fraction)

tok_train = tok_train[:n_train]
tok_val = tok_val[:n_val]
tok_test = tok_test[:n_test]

print(f"Dataset sizes:")
print(f"  Train: {len(tok_train):_} ({len(tok_train)/n_filtered:.2%})")
print(f"  Val:   {len(tok_val):_} ({len(tok_val)/n_filtered:.2%})")
print(f"  Test:  {len(tok_test):_} ({len(tok_test)/n_filtered:.2%})")
print(f"  Total: {len(tok_train) + len(tok_val) + len(tok_test):_}")

Dataset sizes:
  Train: 3_983 (2.10%)
  Val:   853 (0.45%)
  Test:  853 (0.45%)
  Total: 5_689


### Add task prefix to the set

In [11]:
print(f"{CONFIG.task_prefix = }")

train = [
    (tokenizer.decode(sample["input_ids"]), tokenizer.decode(sample["labels"]))
    for sample in tok_train
]
val = [
    (tokenizer.decode(sample["input_ids"]), tokenizer.decode(sample["labels"]))
    for sample in tok_val
]
test = [
    (tokenizer.decode(sample["input_ids"]), tokenizer.decode(sample["labels"]))
    for sample in tok_test
]

CONFIG.task_prefix = 'translate from Spanish to Esperanto: '


Add to each test a 5 random samples

In [12]:
import numpy as np

def get_samples(dataset, seed=0):
    samples = []
    for i in np.random.choice(len(train), CONFIG.num_shots, replace=False):
        source_text, target_text = dataset[i]
        samples.append(f"<es>{source_text}</es><epo>{target_text}</epo>")
    return "\n".join(samples)

test_prompts = [{
    "prompt":f"{CONFIG.task_prefix}\n{get_samples(train, idx)}\n <es>{source_text}</es><epo>",
    "target": target_text
    }
    for idx, (source_text, target_text) in enumerate(test)
]

test_prompts[:5]

[{'prompt': 'translate from Spanish to Esperanto: \n<es><s> es miembro del club de roma</es><epo><s> kaj li estas membro de la roma klubo.</epo>\n<es><s> la parte de enfrente extrema de endemism es la distribución cosmopolita.</es><epo><s> la ekstremaĵkontraŭo de endemio estas kosmopolita distribuo.</epo>\n<es><s> no puedes interferir en eso.</es><epo><s> vi ne povas tusxi tion</epo>\n<es><s> [el embajador está siendo retenido en otra habitación.</es><epo><s> [8] la virinoj estis prenitaj en alian ĉambron.</epo>\n<es><s> 11- como perro que vuelve a su vómito.</es><epo><s> 11 kiel hundo revenas al sia vomitaĵo,</epo>\n <es><s> dos añadas, una noble idea</es><epo>',
  'target': '<s> de la dua flanko, krom nobla ideo'},
 {'prompt': 'translate from Spanish to Esperanto: \n<es><s> es como navegar sin tener</es><epo><s> kiel navigisto sen veloj</epo>\n<es><s> quiero ver a “noah.”</es><epo><s> do, mi devas skribi "na\'vi".</epo>\n<es><s> de la extrema derecha cubanoamericana en la política ag

max token lenght as:
$$
    max(\text{input with samples}) + 2\times mean(\text{base input})
$$

In [ ]:
computed_max_tok = int(
    max([
        len(tokenizer.encode(sample["prompt"]))
        for sample in test_prompts
    ]) + 
    2 * sum(
        len(tokenizer.encode(inputs))
        for inputs in test
    )/len(test)
)
print(f"Computed max token length for prompting: {computed_max_tok:.0f}")

def tokenize_test_prompt(prompt: str):
    return tokenizer(
        prompt,
        max_length=computed_max_tok,
        padding='max_length',
        padding_side='left',
        truncation=True,
    )

model_input = tokenize_test_prompt(test_prompts[0]['prompt'])
print(model_input)
print(tokenizer.batch_decode([model_input['input_ids']]))

Computed max token length for prompting: 425
{'input_ids': tensor([[    2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2, 

In [14]:
from src.data import TranslationDataset

test_prompts = [
    {
        "tokenized": tokenize_test_prompt(sample['prompt']),
        **sample
    }
    for sample in test_prompts
]

shapes = [t["tokenized"]["input_ids"].shape for t in test_prompts[:10]]
print(f"First 10 input_ids shapes: {shapes}")
# dataloader_train = TranslationDataset(
#     tokenized_list=tok_train,
# )
# dataloader_val = TranslationDataset(
#     tokenized_list=tok_val,
# )
dataloader_test = TranslationDataset(
    tokenized_list=[sample["tokenized"] for sample in test_prompts],
)

First 10 input_ids shapes: [torch.Size([1, 425]), torch.Size([1, 425]), torch.Size([1, 425]), torch.Size([1, 425]), torch.Size([1, 425]), torch.Size([1, 425]), torch.Size([1, 425]), torch.Size([1, 425]), torch.Size([1, 425]), torch.Size([1, 425])]


# Load transformer

In [15]:
import torch
from transformers import BitsAndBytesConfig
from transformers import AutoModelForCausalLM

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)


model = AutoModelForCausalLM.from_pretrained(
    CONFIG.model_name,
    token=True,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
)


Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.80s/it]


# Evaluation

In [16]:
from evaluate import load

metric_bleu = load("sacrebleu")
metric_comet = load("comet")

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 6452.78it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/pytorch_light

In [17]:
import numpy as np
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_metrics(preds, labels, sources):

    # Convert to lists if coming from a datasets.Column
    if not isinstance(labels, list):
        labels = list(labels)
        
    if isinstance(preds, tuple):
        preds = preds[0]
    
    # Decode predictions
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace negative ids in the labels as we can't decode them.
    labels = [
        [tokenizer.pad_token_id if j < 0 else j for j in label]
        for label in labels
    ]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Decode sources
    decoded_sources = tokenizer.batch_decode(sources, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result_blue = metric_bleu.compute(
        predictions=decoded_preds, 
        references=decoded_labels
    )
    result_comet = metric_comet.compute(
        sources=decoded_sources,
        predictions=decoded_preds, 
        references=[label[0] for label in decoded_labels]  # COMET expects flat list, not nested
    )
    result = {
        "bleu": result_blue["score"],
        "comet": result_comet["mean_score"]
    }

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

# Inference

In [18]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(
    CONFIG.model_name,
)

print(generation_config)

GenerationConfig {
  "bos_token_id": 1,
  "do_sample": true,
  "eos_token_id": 2,
  "max_length": 4096,
  "pad_token_id": 0,
  "temperature": 0.6,
  "top_p": 0.9
}



In [ ]:
from torch.utils.data import DataLoader
import torch

def collate_fn(batch):
    """Custom collate function to convert lists to tensors"""
    return {
        'input_ids': torch.tensor([item['input_ids'] for item in batch]),
        'attention_mask': torch.tensor([item['attention_mask'] for item in batch]),
        'labels': torch.tensor([item['labels'] for item in batch])
    }

test_batch_size = 32
test_loader = DataLoader(dataloader_test, batch_size=test_batch_size, shuffle=False, collate_fn=collate_fn)

In [20]:
output_sequences = []
all_labels = []
all_sources = []

for i, batch in enumerate(test_loader):
    # Store source input_ids for later decoding
    all_sources.extend(batch['input_ids'].cpu())
    
    # Generate translations
    with torch.no_grad():    
        output_batch = model.generate(
            generation_config=generation_config, 
            input_ids=batch['input_ids'].cuda(), 
            attention_mask=batch['attention_mask'].cuda(), 
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(CONFIG.tgt_code), 
            max_length=CONFIG.max_tok_length, 
            num_beams=1, 
            do_sample=False,
        )
    output_sequences.extend(output_batch.cpu())
    all_labels.extend(batch['labels'].cpu())
    
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(test_loader)} batches")
        break

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ValueError: `attention_mask` passed to `generate` must be 2D.

In [ ]:
result = compute_metrics((output_sequences, all_labels, all_sources))
print(f'BLEU score: {result["bleu"]}')
print(f'COMET score: {result["comet"]}')

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return _C._get_float32_matmul_precision()
You a

BLEU score: 0.3277
COMET score: 0.3101


# Examples

In [ ]:
from maikol_utils.print_utils import print_separator
from src.utils import decode_list, clean_sequence

decoded_sources = decode_list(tokenizer, all_sources)
decoded_outputs = decode_list(tokenizer, output_sequences)

for source, output in zip(decoded_sources, decoded_outputs):
    print(clean_sequence(source))
    print(clean_sequence(output))
    print_separator()

 translate from Spanish to Esperanto: ¡instaura nuevos há
 translate from Spanish to Esperanto: ¡instaura nuevos hábitos!<0x0A> nobody is perfect, but we can all try to be better
________________________________________________________________
 translate from Spanish to Esperanto: abran sus ojos, ab
 translate from Spanish to Esperanto: abran sus ojos, abran sus ojos<0x0A> Hinweis: Esperanto ist eine fiktive Sprache
________________________________________________________________
 translate from Spanish to Esperanto: iremos mal.
 translate from Spanish to Esperanto: iremos mal. Hinweis: Die Übersetzung der Phrase "we're going to go
________________________________________________________________
 translate from Spanish to Esperanto: muchos paquetes de linux tienen
 translate from Spanish to Esperanto: muchos paquetes de linux tienen un sistema de paquetes que es muy complejo y difícil de
________________________________________________________________
 translate from Spanish to Esperan